In [3]:
import time
import yfinance as yf

# After downloading data:
df = yf.download("BANKBEES.NS", period="1y")

# Add this check right after download:
if df.empty:
    st.error("⚠️ Could not fetch data from Yahoo Finance (rate limited or no data). Please wait a moment and reload.")
    st.stop()

/tmp/ipykernel_4591/2290951664.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("BANKBEES.NS", period="1y")
[*********************100%***********************]  1 of 1 completed


In [4]:
import time
import yfinance as yf

for attempt in range(3):
    df = yf.download("BANKBEES.NS", period="1y")
    if not df.empty:
        break
    time.sleep(5)  # wait 5s before retry

if df.empty:
    st.error("⚠️ Yahoo Finance is rate limiting requests. Try again in a minute.")
    st.stop()

/tmp/ipykernel_4591/774974311.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("BANKBEES.NS", period="1y")
[*********************100%***********************]  1 of 1 completed


In [7]:
!pip install streamlit
import yfinance as yf
import streamlit as st

@st.cache_data(ttl=3600)  # cache for 1 hour
def load_data(ticker):
    return yf.download(ticker, period="1y")

df = load_data("BANKBEES.NS")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.6 MB/s eta 0:00:00


2026-05-12 18:04:00.805 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-12 18:04:00.807 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-12 18:04:00.809 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/tmp/ipykernel_4591/2614638407.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return yf.download(ticker, period="1y")
[*********************100%***********************]  1 of 1 completed


In [8]:
@st.cache_data(ttl=3600)
def load_data(ticker):
    import time
    for attempt in range(3):
        df = yf.download(ticker, period="1y")
        if not df.empty:
            return df
        time.sleep(5)
    return df  # return empty df if all retries fail

df = load_data("BANKBEES.NS")

if df.empty:
    st.error("⚠️ Could not load BANKBEES.NS data. Yahoo Finance may be rate limiting. Please reload in a minute.")
    st.stop()

2026-05-12 18:04:22.985 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-12 18:04:22.993 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-12 18:04:22.998 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/tmp/ipykernel_4591/2309527985.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1y")
[*********************100%***********************]  1 of 1 completed


In [9]:
import streamlit as st
import pandas as pd
import numpy as np
import yfinance as yf
from plotly import graph_objs as go
from sklearn.linear_model import LinearRegression
from datetime import date, timedelta

# --- Page Configuration ---
st.set_page_config(page_title="BankBeES Stock Predictor", page_icon="📈")
st.title("📈 BankBeES Stock Price Predictor")
st.markdown("Analyze historical trends and predict future prices.")

# --- Sidebar Configuration ---
st.sidebar.header("Configuration")
stock_symbol = st.sidebar.text_input("Enter Stock Ticker", "BANKBEES.NS")
start_date = "2015-01-01"
today = date.today().strftime("%Y-%m-%d")

# --- Function to Load Data ---
@st.cache_data(ttl=3600)
def load_data(ticker):
    df = yf.download(ticker, start=start_date, end=today)
    if df.empty:
        return None
    df.reset_index(inplace=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

# Fetch data
data = load_data(stock_symbol)

# --- Error Check ---
if data is None:
    st.error("⚠️ Could not fetch data. Yahoo Finance may be rate limiting. Please reload in a minute.")
    st.stop()

# --- 1. Historical Data Display ---
st.subheader("Historical Data Overview (Last 5 Days)")
st.write(data.tail())

# --- 2. Historical Price Chart ---
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=data['Date'], y=data['Close'], name="Close Price", line=dict(color='royalblue')))
fig1.layout.update(title_text="Historical Price Trend", xaxis_rangeslider_visible=True, template="plotly_dark")
st.plotly_chart(fig1)

# --- 3. Machine Learning Prediction ---
st.subheader("Stock Price Prediction")

# Data preparation
data['Date_Ordinal'] = pd.to_datetime(data['Date']).map(pd.Timestamp.toordinal)
X = data['Date_Ordinal'].values.reshape(-1, 1)
y = data['Close'].values.reshape(-1, 1)

# Train model
model = LinearRegression()
model.fit(X, y)

# Predict next 30 days
last_date = data['Date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
future_ordinal = np.array([d.toordinal() for d in future_dates]).reshape(-1, 1)
predictions = model.predict(future_ordinal)

# Display Forecast Data
forecast_df = pd.DataFrame({'Date': future_dates, 'Predicted Price': predictions.flatten()})
st.write("Predicted Prices for the next 30 days:")
st.dataframe(forecast_df.head(10))

# --- 4. Forecast Chart ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=data['Date'], y=data['Close'], name="Historical Close", line=dict(color='gray')))
fig2.add_trace(go.Scatter(x=forecast_df['Date'], y=forecast_df['Predicted Price'], name="Predicted Price", line=dict(color='orange', dash='dash')))
fig2.layout.update(title_text="Price Forecast (Next 30 Days)", template="plotly_dark")
st.plotly_chart(fig2)

st.sidebar.markdown("---")
st.sidebar.info("Educational purposes only. No financial advice.")

2026-05-12 18:05:08.246 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:08.248 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:08.454 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-12 18:05:08.455 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:08.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:08.459 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:08.461 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [10]:
# --- 4. Forecast Chart ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=data['Date'], y=data['Close'], name="Historical Close", line=dict(color='gray')))
fig2.add_trace(go.Scatter(x=forecast_df['Date'], y=forecast_df['Predicted Price'], name="Predicted Price", line=dict(color='orange', dash='dash')))
fig2.layout.update(title_text="Price Forecast (Next 30 Days)", template="plotly_dark")
st.plotly_chart(fig2)

2026-05-12 18:05:36.950 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:36.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:36.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:36.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:05:36.960 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [11]:
import streamlit as st
import pandas as pd
import numpy as np
import yfinance as yf
from plotly import graph_objs as go
from sklearn.linear_model import LinearRegression
from datetime import date, timedelta

# --- Page Configuration ---
st.set_page_config(page_title="BankBeES Stock Predictor", page_icon="📈")
st.title("📈 BankBeES Stock Price Predictor")
st.markdown("Analyze historical trends and predict future prices.")

# --- Sidebar Configuration ---
st.sidebar.header("Configuration")
stock_symbol = st.sidebar.text_input("Enter Stock Ticker", "BANKBEES.NS")
start_date = "2015-01-01"
today = date.today().strftime("%Y-%m-%d")

# --- Load Data ---
@st.cache_data(ttl=3600)
def load_data(ticker):
    df = yf.download(ticker, start=start_date, end=today)
    if df.empty:
        return None
    df.reset_index(inplace=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

data = load_data(stock_symbol)

if data is None:
    st.error("Could not fetch data. Yahoo Finance may be rate limiting. Please reload.")
    st.stop()

# --- Historical Data ---
st.subheader("Historical Data Overview (Last 5 Days)")
st.write(data.tail())

# --- Historical Chart ---
fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(x=data['Date'], y=data['Close'], name="Close Price", line=dict(color='royalblue'))
)
fig1.layout.update(
    title_text="Historical Price Trend",
    xaxis_rangeslider_visible=True,
    template="plotly_dark"
)
st.plotly_chart(fig1)

# --- Prediction ---
st.subheader("Stock Price Prediction")

data['Date_Ordinal'] = pd.to_datetime(data['Date']).map(pd.Timestamp.toordinal)
X = data['Date_Ordinal'].values.reshape(-1, 1)
y = data['Close'].values.reshape(-1, 1)

model = LinearRegression()
model.fit(X, y)

last_date = data['Date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
future_ordinal = np.array([d.toordinal() for d in future_dates]).reshape(-1, 1)
predictions = model.predict(future_ordinal)

forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted Price': predictions.flatten()
})

st.write("Predicted Prices for the next 30 days:")
st.dataframe(forecast_df.head(10))

# --- Forecast Chart ---
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(x=data['Date'], y=data['Close'], name="Historical Close", line=dict(color='gray'))
)
fig2.add_trace(
    go.Scatter(x=forecast_df['Date'], y=forecast_df['Predicted Price'], name="Predicted Price", line=dict(color='orange', dash='dash'))
)
fig2.layout.update(
    title_text="Price Forecast (Next 30 Days)",
    template="plotly_dark"
)
st.plotly_chart(fig2)

st.sidebar.markdown("---")
st.sidebar.info("Educational purposes only. No financial advice.")

2026-05-12 18:06:03.383 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.388 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.391 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.392 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.393 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:06:03.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [12]:
import streamlit as st
import pandas as pd
import numpy as np
import yfinance as yf
from plotly import graph_objs as go
from sklearn.linear_model import LinearRegression
from datetime import date, timedelta

st.set_page_config(page_title="BankBeES Stock Predictor", page_icon="📈")
st.title("📈 BankBeES Stock Price Predictor")
st.markdown("Analyze historical trends and predict future prices.")

st.sidebar.header("Configuration")
stock_symbol = st.sidebar.text_input("Enter Stock Ticker", "BANKBEES.NS")
start_date = "2015-01-01"
today = date.today().strftime("%Y-%m-%d")

@st.cache_data(ttl=3600)
def load_data(ticker):
    df = yf.download(ticker, start=start_date, end=today)
    if df.empty:
        return None
    df.reset_index(inplace=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

data = load_data(stock_symbol)

if data is None:
    st.error("Could not fetch data. Please reload after 1 minute.")
    st.stop()

st.subheader("Historical Data (Last 5 Days)")
st.write(data.tail())

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=data['Date'],
    y=data['Close'],
    name="Close Price",
    line=dict(color='royalblue')
))
fig1.update_layout(
    title="Historical Price Trend",
    xaxis_rangeslider_visible=True,
    template="plotly_dark"
)
st.plotly_chart(fig1)

st.subheader("Stock Price Prediction (Next 30 Days)")

data['Date_Ordinal'] = pd.to_datetime(data['Date']).map(pd.Timestamp.toordinal)
X = data['Date_Ordinal'].values.reshape(-1, 1)
y = data['Close'].values.reshape(-1, 1)

model = LinearRegression()
model.fit(X, y)

last_date = data['Date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
future_ordinal = np.array([d.toordinal() for d in future_dates]).reshape(-1, 1)
predictions = model.predict(future_ordinal)

forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted Price': predictions.flatten()
})

st.write("Predicted Prices:")
st.dataframe(forecast_df.head(10))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=data['Date'],
    y=data['Close'],
    name="Historical Close",
    line=dict(color='gray')
))
fig2.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Predicted Price'],
    name="Predicted Price",
    line=dict(color='orange', dash='dash')
))
fig2.update_layout(
    title="Price Forecast (Next 30 Days)",
    template="plotly_dark"
)
st.plotly_chart(fig2)

st.sidebar.markdown("---")
st.sidebar.info("Educational purposes only. No financial advice.")

2026-05-12 18:07:08.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.705 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.707 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.711 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:07:08.715 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())